In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection

# ── Fonts ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

# ── Thesis colour palette ─────────────────────────────────────────────────────
TROPICAL = (0.0, 0.46, 0.37)        # tropicalrainforest
CTRED    = (177/255, 56/255, 69/255) # CTRed

LABEL_GREY = (0.6, 0.6, 0.6)        # panel-label grey

def _pale(rgb, mix=0.60):
    return tuple(c + (1 - c) * mix for c in rgb)

CMAPS = {
    "diffusive":      mcolors.LinearSegmentedColormap.from_list(
                          "diff",  [_pale(TROPICAL), TROPICAL]),
    "superdiffusive": mcolors.LinearSegmentedColormap.from_list(
                          "super", [_pale(CTRED), CTRED]),
}

# ── ERW simulator ─────────────────────────────────────────────────────────────
def simulate_erw(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past
    S = np.empty(n + 1, dtype=np.int64)
    S[0] = 0
    S[1:] = np.cumsum(xi[1:])
    return S

def gradient_path(ax, x, y, cmap, vmax, lw=0.45, alpha=0.88):
    pts  = np.array([x, y], dtype=float).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    vals = (np.abs(y[:-1]) + np.abs(y[1:])) / 2.0
    norm = mcolors.Normalize(vmin=0, vmax=vmax)
    lc   = LineCollection(segs, cmap=cmap, norm=norm,
                          linewidth=lw, alpha=alpha, rasterized=True)
    lc.set_array(vals)
    ax.add_collection(lc)

# ── Parameters ────────────────────────────────────────────────────────────────
N_STEPS  = 800
N_PATHS  = 8
Q        = 0.5

REGIMES = [
    ("diffusive",      0.30, r"$p = 0.30$"),
    ("superdiffusive", 0.80, r"$p = 0.80$"),
]

steps = np.arange(N_STEPS + 1)
rng   = np.random.default_rng(42)

# ── Simulate ──────────────────────────────────────────────────────────────────
all_paths = {}
for regime, p_val, _ in REGIMES:
    all_paths[regime] = [simulate_erw(N_STEPS, p=p_val, q=Q, rng=rng)
                         for _ in range(N_PATHS)]

# ── Shared y-limits ───────────────────────────────────────────────────────────
ylim = (-150, 150)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(5.0, 2.3), sharey=True)
fig.subplots_adjust(left=0.10, right=0.97, bottom=0.18, top=0.88, wspace=0.18)

for ax, (regime, p_val, label) in zip(axes, REGIMES):
    cmap  = CMAPS[regime]
    paths = all_paths[regime]
    # colour scale matched to fixed y-range
    vmax  = 150

    for S in paths:
        gradient_path(ax, steps, S, cmap=cmap, vmax=vmax)

    ax.set_xlim(0, N_STEPS)
    ax.set_ylim(*ylim)

    ax.axhline(0, color="black", lw=0.45, ls=(0, (4, 3)), alpha=0.28, zorder=0)

    # panel label in thesis grey
    ax.set_title(label, fontsize=10, pad=5, color=LABEL_GREY)
    ax.set_xlabel(r"$n$", labelpad=3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _, ns=N_STEPS: f"${int(x)}$"
    ))
    ax.yaxis.set_major_locator(ticker.FixedLocator([-100, 0, 100]))

    if ax is axes[0]:
        ax.set_ylabel(r"$S_n$", labelpad=4)
plt.close()
print("Done")